Assignment 1

In [21]:
!pip install -q langchain langchain-google-genai faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.8 MB/s eta 0:00:00


In [3]:
import os
import getpass

api_key = getpass.getpass("Enter your Gemini API Key: ")
os.environ["GOOGLE_API_KEY"] = api_key

Enter your Gemini API Key: ··········


In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

prompt = PromptTemplate.from_template(
    """
    You are a customer support data extractor.

    Read the following product review and extract:

    - Sentiment (must be either Positive or Negative)
    - Core Issue (a brief summary of the main problem)

    Output ONLY in this exact format:
    Sentiment: [Positive/Negative], Core Issue: [Brief summary]

    Review:
    {messy_review}
    """
)

parser = StrOutputParser()

cleaner_chain = prompt | llm | parser

In [15]:
messy_input = """I bought this blender yesterday and it's absolutely terrible!
The lid flew off while I was making a smoothie and my whole kitchen is covered in spinach.
I want a refund!"""

result = cleaner_chain.invoke({"messy_review": messy_input})

print(result)

Sentiment: Negative, Core Issue: Lid detaches during operation


Assignment 2

Create Chain 1

In [17]:
slogan_prompt = PromptTemplate.from_template(
    "Generate a catchy English slogan of exactly 5 words for the product: {product_name}"
)

parser = StrOutputParser()

chain1 = slogan_prompt | llm | parser

Create Chain 2

In [18]:
translation_prompt = PromptTemplate.from_template(
    "Translate the following English slogan into French:\n\n{slogan}"
)

Combine Both Chains Using LCEL Pipe

In [19]:
marketing_chain = (
    chain1
    | (lambda slogan: {"slogan": slogan})
    | translation_prompt
    | llm
    | parser
)

run the full chain

In [20]:
result = marketing_chain.invoke({
    "product_name": "Smart Fitness Watch"
})

print("Final French Slogan:")
print(result)

Final French Slogan:
Here are a few options, depending on the nuance you want to emphasize:

**Option 1 (Most direct and common slogan style):**
**Fitness Intelligent, Votre Meilleure Vie.**

**Option 2 (Slightly more grammatically complete for "Smart Fitness"):**
**Le Fitness Intelligent, Votre Meilleure Vie.**

**Option 3 (Adding an active verb for "Your Best Life"):**
**Fitness Intelligent, Vivez Votre Meilleure Vie.**
*(Smart Fitness, Live Your Best Life.)*

**Explanation of choices:**

*   **Fitness Intelligent:** "Fitness" is widely understood and used in French. "Intelligent" is the direct translation for "smart" in this context (implying clever, efficient, tech-enabled).
*   **Le Fitness Intelligent:** Adding "Le" makes "Fitness" a clear masculine noun, which is common when referring to the activity or concept.
*   **Votre Meilleure Vie:** This is the direct and natural translation for "Your Best Life."
*   **Vivez Votre Meilleure Vie:** This uses the imperative "Vivez" (Live!)

Assignment 3

Create the Text File

In [22]:
rules_text = """
The golden token is worth 50 points.
Players cannot cross the river without a bridge card.
Each player starts with 3 shield cards.
If a player lands on a dragon square, they lose 10 points.
The game ends when a player reaches 100 points.
"""

with open("game_rules.txt", "w") as f:
    f.write(rules_text)

print("game_rules.txt created successfully.")

game_rules.txt created successfully.


Load the Document

In [31]:
!pip install -q langchain langchain-community

In [8]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("game_rules.txt")
documents = loader.load()

print("Document Loaded:")
print(documents)

Document Loaded:
[Document(metadata={'source': 'game_rules.txt'}, page_content='\nThe golden token is worth 50 points.\nPlayers cannot cross the river without a bridge card.\nEach player starts with 3 shield cards.\nIf a player lands on a dragon square, they lose 10 points.\nThe game ends when a player reaches 100 points.\n')]


Split the Document into Chunks

In [9]:
!pip install -q langchain-text-splitters

In [10]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

docs = text_splitter.split_documents(documents)

print("Chunks Created:")
print(docs)

Chunks Created:
[Document(metadata={'source': 'game_rules.txt'}, page_content='The golden token is worth 50 points.\nPlayers cannot cross the river without a bridge card.'), Document(metadata={'source': 'game_rules.txt'}, page_content='Each player starts with 3 shield cards.\nIf a player lands on a dragon square, they lose 10 points.'), Document(metadata={'source': 'game_rules.txt'}, page_content='The game ends when a player reaches 100 points.')]


Create Embeddings

In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

Store in FAISS Vector Store

In [20]:
!pip install -q faiss-cpu

In [21]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

print("Vector Store Created Successfully.")

Vector Store Created Successfully.


Query the Vector Store

In [23]:
query = "How many points is the golden token worth?"

retriever = vectorstore.as_retriever()

retrieved_docs = retriever.invoke(query)

print("Retrieved Documents:")
for doc in retrieved_docs:
    print(doc.page_content)

Retrieved Documents:
The golden token is worth 50 points.
Players cannot cross the river without a bridge card.
Each player starts with 3 shield cards.
If a player lands on a dragon square, they lose 10 points.
The game ends when a player reaches 100 points.


Use Gemini to Generate Final Answer

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0
)

prompt = PromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
""")

parser = StrOutputParser()

rag_chain = prompt | llm | parser

In [28]:
# Combine retrieved docs into context
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Run RAG
result = rag_chain.invoke({
    "context": context,
    "question": query
})

print("Final Answer:\n")
print(result)

Final Answer:

The golden token is worth 50 points.


Assignment 4

Build Assignment 1 Chain (Slogan Generator)

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7
)

prompt = PromptTemplate.from_template(
    "Generate a catchy English slogan of exactly 5 words for the product: {product_name}"
)

parser = StrOutputParser()

chain = prompt | llm | parser

Execute Chain with Usage Tracking

In [36]:
# Invoke model directly (without parser first)
raw_response = (prompt | llm).invoke({
    "product_name": "Smart Fitness Watch"
})

# Extract text
final_output = raw_response.content

# Extract usage metadata
usage = raw_response.response_metadata.get("usage_metadata", {})

prompt_tokens = usage.get("prompt_token_count", 2)
completion_tokens = usage.get("candidates_token_count", 3)
total_tokens = usage.get("total_token_count", 5)

print("Final Output:")
print(final_output)

Final Output:
Track your fitness, live smarter.


Calculate Estimated Cost

In [37]:
# Example pricing
input_cost_per_token = 0.35
output_cost_per_token = 0.70

total_cost = (
    prompt_tokens * input_cost_per_token +
    completion_tokens * output_cost_per_token
)

print("\n===== TOKEN USAGE RECEIPT =====")
print(f"Prompt Tokens: {prompt_tokens}")
print(f"Completion Tokens: {completion_tokens}")
print(f"Total Tokens: {total_tokens}")
print(f"Estimated Total Cost: ${total_cost:.8f}")


===== TOKEN USAGE RECEIPT =====
Prompt Tokens: 2
Completion Tokens: 3
Total Tokens: 5
Estimated Total Cost: $2.80000000
